In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import uuid

import pandas as pd

from naturecubepy import (
    auth_headers,
    get_key,
    get_procedure,
    get_project,
    get_project_systems,
    list_systems,
    upload_observations_from_csv,
    validate_csv_against_procedure,
)


In [13]:
api_key = get_key('EV_NI_SIT')
hdr = auth_headers(api_key, okala_url="https://sit.naturecube.io/api")
project_name = get_project(hdr)

Retrieving project data...
Received response with status code 200
Project data retrieved successfully
Setting your active project as - NI TEST


In [14]:
# Fetch systems/procedures for this project, then select the procedure that matches your CSV.
project_systems = get_project_systems(hdr)
systems_table = list_systems(project_systems)

 system_index                    system_name  system_id procedure_index           procedure_name procedure_id  form
            1   Song Meter Mini Bat 2 Li-ion        338            <NA>                      NaN         <NA>  <NA>
            2 Browning Recon Force Elite HP5        337               1     Environmental sample          157 False
            2 Browning Recon Force Elite HP5        337               2 Simple sensor deployment          209 False
            2 Browning Recon Force Elite HP5        337               3               Harsh Test          259 False
            2 Browning Recon Force Elite HP5        337               4            One Procedure          309 False
            2 Browning Recon Force Elite HP5        337               5           Procedure 1:21          358 False
            2 Browning Recon Force Elite HP5        337               6            Procedure One          407 False
            2 Browning Recon Force Elite HP5        337               7 

In [15]:
procedure = get_procedure(
    project_systems,
    system_id=460,
    procedure_id=860
)
print(f"Using system_id={procedure['system_id']}, procedure_id={procedure['procedure_id']}")


System: bird survey (id: 460)
Procedure: bird survey (id: 860, form: False)
Items (3):
item_id                            item_uuid       item_name item_description data_type  nullable                                                                                                 choices
   1645 d8749bbe-cbfc-47a6-9381-f01a716065cd           Taxon            label     label      True                                                                                                        
   1646 8ed11b87-780d-48b6-817f-f2934a310a10        behavior         behavior    choice      True singing | calling | alarm | family | pair | fly.over | land | fly.away | carry.nest | carry.food | nest
   <NA> f47ac10b-58cc-4372-a567-0e02b2c3d479 Taxonomic label  Taxonomic label     label      True                                                                                                        
Using system_id=460, procedure_id=860


In [23]:
CSV_PATH =  Path("/Users/natimi/Projects/BioacousticsPaper/data/RawDataBirds_wren2015_naturecube.csv")

print(f"CSV path: {CSV_PATH}")
print(CSV_PATH.exists())

validation = validate_csv_against_procedure(
    procedure=procedure,
    csv_path=CSV_PATH,
    
)
print("valid:", validation["valid"])
if validation["issues"]:
    print("issues:")
    for issue in validation["issues"]:
        print("-", issue)

In [24]:
# Multi-select choice cells (e.g. calling;land) upload as a list on one observation.
# No expansion step needed — validate and upload the source CSV directly.
UPLOAD_CSV_PATH = CSV_PATH
print(f"Upload CSV: {UPLOAD_CSV_PATH}")


In [25]:
validation = validate_csv_against_procedure(procedure=procedure, csv_path=UPLOAD_CSV_PATH)
print("valid:", validation["valid"])

In [26]:
upload_result = upload_observations_from_csv(
    hdr,
    UPLOAD_CSV_PATH,
    procedure=procedure,
    recorded_at_format="%Y-%m-%dT%H:%M:%S",
    dry_run=False,
)
print(f"{upload_result['succeeded']} uploaded, {upload_result['failed']} failed")
if upload_result["rejected"]:
    display(pd.DataFrame(upload_result["rejected"]))


Uploading batch 1/19 (500 observations, 500/9029 total)...
Uploading batch 2/19 (500 observations, 1000/9029 total)...


KeyboardInterrupt: 

In [22]:
RECORDED_AT_FORMAT = "%Y-%m-%dT%H:%M:%S"

upload_result = upload_observations_from_csv(
    hdr=hdr,
    csv_path=CSV_PATH,
    procedure=procedure,
    dry_run=False,
    recorded_at_format=RECORDED_AT_FORMAT,
)
print("uploaded:", upload_result["uploaded"])
print(upload_result.get("response"))


Uploading batch 1/1 (19 observations, 19/19 total)...
uploaded: True
[{'survey_uuid': '4efc5830-cb8b-4011-ba9a-52edf0b43b06', 'status': 'error', 'message': "taxonomy label 'bogus label 1' was not found; use a valid iucn_labels.label name or label_id"}, {'survey_uuid': '32989cc1-d651-4b02-99eb-cf2599fdecde', 'status': 'error', 'message': "taxonomy label 'bogus label 1' was not found; use a valid iucn_labels.label name or label_id"}, {'survey_uuid': '4c4d8b8f-9702-48e7-833d-6b068caf93ce', 'status': 'error', 'message': "taxonomy label 'bogues label 2' was not found; use a valid iucn_labels.label name or label_id"}, {'survey_uuid': 'edc707eb-8b9c-4ba9-8049-e2c889ff2423', 'status': 'success', 'message': None}, {'survey_uuid': '1c68ab51-9c3e-4b8a-ba7c-6887e7ff6258', 'status': 'success', 'message': None}, {'survey_uuid': 'b2c31a00-38d6-4987-917e-776785ccb6ec', 'status': 'success', 'message': None}, {'survey_uuid': 'b4a60db0-c0c7-4724-a0be-a34f1f370cd0', 'status': 'success', 'message': None}, 